# WildSpeech M8.3 — Exhaustive Performance Bottleneck Laboratory

**Question:** Why have M8 iterations produced only marginal improvements? This notebook audits **data provenance → feature coverage → target geometry → matched-fold ablations → school generalization → residual predictability → pair ranking → label contradictions → train/test drift → prioritized next experiment**.

**Evidence policy:** Target labels are **train only**. Fold-fitted transforms and student- and school-aware out-of-fold (OOF) evaluation are used for predictive evidence. Score correlations, nearest-neighbor gaps and clusters are **descriptive only**. Private child-level predictions remain in Google Drive, never GitHub. No raw speech is downloaded or re-extracted.

**Run:** Execute **all cells** with `MODE="SMOKE"` (first), inspect Drive/cache checks, then change to `MODE="FULL"` and run all cells again.

## 0 — Google Drive mount (first)

The official competition dataset and previously extracted M7/M8 features live in **your Drive**. Colab mounts `/content/drive/MyDrive`. Local Jupyter can instead use `WILDSPEECH_DRIVE_ROOT`.

In [ ]:
import sys, os, json, re, math, time, hashlib, warnings, subprocess, importlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML, Image
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE=Path("/content/drive/MyDrive")
else:
    DRIVE=Path(os.environ.get("WILDSPEECH_DRIVE_ROOT",str(Path.home()/"MyDrive"))).expanduser()
assert DRIVE.is_dir(), f"Drive is not mounted: {DRIVE}"
print("Drive:",DRIVE, "| Colab:",IN_COLAB)
print("Matching top-level folders:",sorted(p.name for p in DRIVE.iterdir()
      if p.is_dir() and any(k in p.name.lower() for k in ("wild","speech")))[:30])

## 1 — Install the GitHub pipeline directly (no ZIP or uploads)

The notebook pulls `main` by fast-forward, installs `WildSpeech_M8` editable, and loads the M8.3 diagnostic engine. It refuses to destructively reset local checkouts.

In [ ]:
GITHUB_URL="https://github.com/mybenkhadda/WildSpeech.git"
CHECKOUT=Path("/content/WildSpeech_git") if IN_COLAB else Path.cwd()/"WildSpeech_git"
REPO=CHECKOUT/"WildSpeech_M8"
if not (CHECKOUT/".git").is_dir():
    if CHECKOUT.exists(): raise RuntimeError(f"{CHECKOUT} exists but is not a Git checkout.")
    subprocess.run(["git","clone","--depth","1","--branch","main",GITHUB_URL,str(CHECKOUT)],check=True)
else:
    subprocess.run(["git","-C",str(CHECKOUT),"pull","--ff-only","origin","main"],check=True)
HEAD_SHA=subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
assert (REPO/"src/wildspeech_m8/m83_bottleneck.py").is_file(),"M8.3 module is missing from checkout."
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],check=True)
source=str(REPO/"src")
if source not in sys.path: sys.path.insert(0,source)
importlib.invalidate_caches()
from wildspeech_m8.m8_pipeline import Config, read_official, prepare_dataset, FoldEncoder
from wildspeech_m8.m83_bottleneck import AuditConfig, run_frame, safe_columns, cluster_bootstrap
print("Source SHA:",HEAD_SHA,"\nM8.3 successfully loaded from GitHub.")

## 2 — Choose the experiment profile and optional Drive paths

`SMOKE` verifies integration and produces **no leaderboard claim**. `FULL` performs the complete experiment. Do not use FULL to interpret acoustic feature value if its cached features are absent or partially extracted.

In [ ]:
MODE="SMOKE"                    # change to FULL for the exhaustive experiment
DATA_ROOT_OVERRIDE=""             # folder holding the six official CSVs; leave blank to discover
OUTPUT_ROOT_OVERRIDE=""           # M8 feature-cache root; leave blank to discover
PROFILE_CSV_OVERRIDE=""           # M7 profile CSV; leave blank to discover
REQUIRE_AUDIO_CACHE_FOR_FULL=True
REQUIRE_SPEECH_CACHE_FOR_FULL=False
REQUIRE_PHONETIC_CACHE_FOR_FULL=False
FORCE_RERUN=False
SEED=2026
MODE=MODE.upper()
assert MODE in ("SMOKE","FULL")
FULL=MODE=="FULL"
N_FOLDS=5 if FULL else 3
N_REPEATS=2 if FULL else 1
MAX_FEATURES=450 if FULL else 220
BOOTSTRAPS=400 if FULL else 60
RUN_DRIFT_AUC=True
RUN_NEIGHBORS=True
print(f"{MODE}: folds={N_FOLDS}, repeats={N_REPEATS}, features≤{MAX_FEATURES}, bootstraps={BOOTSTRAPS}")

## 3 — Discover and validate the official competition dataset in Drive

All six CSVs must exist together; IDs are aligned using the repository loader. Discovery scans *directories only* at a limited depth, never audio archives.

In [ ]:
REQUIRED=("train_student_metadata.csv","test_student_metadata.csv","train_labels.csv",
          "submission_format.csv","train_clip_metadata.csv","test_clip_metadata.csv")
def has_official(p):
    return p.is_dir() and all((p/f).is_file() for f in REQUIRED)
def discover_data():
    if DATA_ROOT_OVERRIDE:
        p=Path(DATA_ROOT_OVERRIDE).expanduser()
        if not has_official(p): raise FileNotFoundError(f"Official CSVs missing from {p}: {[f for f in REQUIRED if not (p/f).is_file()]}")
        return p
    hints=["WildSpeech","WildSpeech/data","WildSpeech/dataset","WildSpeech/competition",
           "WildSpeech/raw","WildSpeech/data/official","WildSpeech_Data","WildSpeech_M8/data"]
    hits=[DRIVE/h for h in hints if has_official(DRIVE/h)]
    if not hits:
        queue=[(DRIVE,0)];visited=0
        while queue and visited<250:
            here,depth=queue.pop(0);visited+=1
            if has_official(here):hits.append(here);continue
            if depth>=3:continue
            try:children=[p for p in here.iterdir() if p.is_dir()]
            except (OSError,PermissionError):continue
            queue.extend((p,depth+1) for p in children if depth==0 or
                         any(token in p.name.lower() for token in ("wild","speech","data","dataset","competition","input")))
    hits=sorted(set(hits),key=str)
    if len(hits)!=1: raise RuntimeError(f"Found {len(hits)} official datasets: {hits}. Set DATA_ROOT_OVERRIDE to the correct folder.")
    return hits[0]
DATA_ROOT=discover_data()
print("Selected official data root:",DATA_ROOT)
for filename in REQUIRED:
    p=DATA_ROOT/filename
    print(" ✓",filename,":",round(p.stat().st_size/1024,1),"KiB")

## 4 — Discover M8 feature caches and the M7 profile from Drive

The M8 loader can fall back to official metadata when the M7 profile is missing. We explicitly expose that condition: without the intended screening baseline or audio cache, conclusions about a performance ceiling are not trustworthy.

In [ ]:
CACHE_FILES={"audio":"audio_student_features.csv","speech":"speech_student_features.csv",
             "phonetic":"phonetic_student_features.csv"}
def n_caches(p):return sum((p/"tables"/v).is_file() for v in CACHE_FILES.values())
if OUTPUT_ROOT_OVERRIDE:
    OUTPUT_ROOT=Path(OUTPUT_ROOT_OVERRIDE).expanduser()
    if not OUTPUT_ROOT.is_dir(): raise FileNotFoundError(OUTPUT_ROOT)
else:
    options=[DRIVE/"WildSpeech_M8",DRIVE/"WildSpeech"/"outputs"/"m8",
             DRIVE/"WildSpeech_M81",DRIVE/"WildSpeech"/"WildSpeech_M8"]
    options.extend(p for p in DRIVE.iterdir() if p.is_dir() and "wildspeech" in p.name.lower())
    available=[p for p in set(options) if p.is_dir()]
    OUTPUT_ROOT=sorted(available,key=lambda p:(n_caches(p),int(p.name=="WildSpeech_M8")),reverse=True)[0] if available else DRIVE/"WildSpeech_M8"
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
profile_options=[]
if PROFILE_CSV_OVERRIDE:
    p=Path(PROFILE_CSV_OVERRIDE).expanduser()
    if not p.is_file():raise FileNotFoundError(p)
    profile_options=[p]
else:
    preferred=[DRIVE/"WildSpeech_M71_PHENOTYPE_BOOSTING"/"tables",
               DRIVE/"WildSpeech_M7_DEEP_EDA"/"tables",OUTPUT_ROOT/"tables",DATA_ROOT]
    preferred += [p/"tables" for p in DRIVE.iterdir() if p.is_dir() and "wildspeech" in p.name.lower()]
    profile_options=list(dict.fromkeys(p/"student_profiles_all_covariates.csv" for p in preferred
                                       if (p/"student_profiles_all_covariates.csv").is_file()))
PROFILE_CSV=str(profile_options[0]) if profile_options else ""
print("Feature-cache root:",OUTPUT_ROOT)
print("M7 profile:",PROFILE_CSV or "MISSING — official metadata fallback!")
if len(profile_options)>1:print("Other valid-looking profiles:",[str(p) for p in profile_options[1:]])
cache_status=[]
for fam,filename in CACHE_FILES.items():
    file=OUTPUT_ROOT/"tables"/filename
    info=OUTPUT_ROOT/"tables"/(fam+"_extraction_summary.json")
    complete=None
    if info.is_file():
        try:complete=json.loads(info.read_text()).get("is_complete")
        except (OSError,ValueError):warnings.warn(f"Unreadable summary: {info}")
    cache_status.append(dict(family=fam,found=file.is_file(),complete=complete,
                             megabytes=round(file.stat().st_size/1e6,2) if file.exists() else 0))
cache_df=pd.DataFrame(cache_status)
display(cache_df)
if FULL:
    required={"audio":REQUIRE_AUDIO_CACHE_FOR_FULL,"speech":REQUIRE_SPEECH_CACHE_FOR_FULL,
              "phonetic":REQUIRE_PHONETIC_CACHE_FOR_FULL}
    missing=[r["family"] for r in cache_status if required[r["family"]] and not r["found"]]
    partial=[r["family"] for r in cache_status if r["found"] and r["complete"] is False]
    if missing or partial:raise RuntimeError(f"FULL M8 cache is missing {missing} or partial {partial}; choose correct OUTPUT_ROOT or rebuild complete features.")
if not PROFILE_CSV:warnings.warn("M7 profile missing: screening comparisons may not match earlier runs.")

## 5 — Validate official joins and target integrity

Read official CSVs through the repository's strict identity-alignment checks. Only train labels are accessed; test outcomes are not available. Output tables contain aggregate counts, never raw student IDs.

In [ ]:
raw_cfg=Config(data_root=str(DATA_ROOT),output_root=str(OUTPUT_ROOT),
               profile_csv=PROFILE_CSV,quick=not FULL,max_features=MAX_FEATURES)
train_meta,test_meta,labels,fmt,clips=read_official(raw_cfg)
assert set(train_meta.subject_id)==set(labels.subject_id)
assert set(test_meta.subject_id)==set(fmt.subject_id)
assert not set(train_meta.subject_id)&set(test_meta.subject_id)
y_raw=labels.outcome_score.to_numpy(float)
assert np.isfinite(y_raw).all()
schools_raw=train_meta["school_id"].fillna("UNKNOWN").astype(str).to_numpy() if "school_id" in train_meta else np.repeat("UNKNOWN",len(train_meta))
source_stats=pd.DataFrame([
    {"dataset":"train students","rows":len(train_meta),"columns":train_meta.shape[1]},
    {"dataset":"test students","rows":len(test_meta),"columns":test_meta.shape[1]},
    {"dataset":"train labels","rows":len(labels),"columns":labels.shape[1]},
    {"dataset":"train clips","rows":len(clips["train"]),"columns":clips["train"].shape[1]},
    {"dataset":"test clips","rows":len(clips["test"]),"columns":clips["test"].shape[1]},
    {"dataset":"submission template","rows":len(fmt),"columns":fmt.shape[1]}])
display(source_stats)
print("Score mean/std:",round(float(np.mean(y_raw)),4),round(float(np.std(y_raw)),4))
print("Score min/max:",round(float(np.min(y_raw)),4),round(float(np.max(y_raw)),4))
print("Distinct scores:",int(pd.Series(y_raw).nunique())," | schools:",len(set(schools_raw)))

## 6 — Define a reusable figure writer on Drive

All figures go to `<OUTPUT_ROOT>/m83_bottleneck/extended_figures/`, and can be re-rendered independently of the slow model run.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from matplotlib.ticker import MaxNLocator
plt.rcParams.update({"figure.dpi":110,"savefig.dpi":155,"axes.grid":True,"grid.alpha":.16})
OUT=OUTPUT_ROOT/"m83_bottleneck"
FIG_DIR=OUT/"extended_figures"
FIG_DIR.mkdir(parents=True,exist_ok=True)
FIGURES=[]
def save_figure(key,title):
    plt.tight_layout()
    path=FIG_DIR/f"{key}.png"
    plt.savefig(path,dpi=155,bbox_inches="tight")
    plt.show()
    plt.close()
    FIGURES.append({"key":key,"title":title,"path":str(path.relative_to(OUT))})
print("Figures will be saved to:",FIG_DIR)

## 7 — Explore target range, ceiling/floor effects, and school imbalance

Observed score ties and school heterogeneity may complicate ranking. They do **not** prove irreducible label noise: a noise estimate requires independent repeated annotations or other assumptions.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].hist(y_raw,bins=min(30,max(8,len(np.unique(y_raw)))),edgecolor="white")
axs[0].set(title="Reading score: training distribution",xlabel="Outcome score",ylabel="Students")
percentiles=[0,5,10,25,50,75,90,95,100]
axs[1].plot(percentiles,np.percentile(y_raw,percentiles),"o-")
axs[1].set(title="Target quantiles",xlabel="Percentile",ylabel="Score")
save_figure("01_score_geometry","Outcome distribution and quantiles")
vc=pd.Series(schools_raw).value_counts()
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(range(len(vc)),vc.values)
ax.set(title="School sizes (identities hidden)",xlabel="Anonymized school index",ylabel="Training students")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
save_figure("02_school_sample_sizes","School sample-size imbalance")
print("Target descriptive summary:",pd.Series(y_raw).describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]).round(3).to_dict())
print("Schools: min students",vc.min(),"max",vc.max(),"median",vc.median())

## 8 — Materialize the student feature bank from official data + M7/M8 caches

`prepare_dataset()` validates IDs, joins existing `tables/speech_student_features.csv`, `audio_student_features.csv`, and `phonetic_student_features.csv`, applies transformations, selects the feature bank, and returns aligned train/test rows. **This step does not read audio waveforms.**

In [ ]:
start=time.perf_counter()
X_train,X_test,y,schools,submission_fmt,clip_data,feature_manifest=prepare_dataset(raw_cfg)
assert len(X_train)==len(y)==len(schools)==len(train_meta)
assert len(X_test)==len(test_meta)
assert list(X_train.columns)==list(X_test.columns)
assert X_train.index.is_unique and X_test.index.is_unique
assert not set(X_train.index)&set(X_test.index)
assert np.isfinite(y).all()
columns,family=safe_columns(X_train,MAX_FEATURES)
assert set(columns)==set(X_train.columns),"Feature registry mismatch. Check M8 feature selection."
print(f"Loaded in {time.perf_counter()-start:.1f}s | train={X_train.shape} test={X_test.shape}")
print("M7 profile:",PROFILE_CSV or "NOT FOUND")
print("Feature-family counts:")
display(feature_manifest.groupby("family").agg(n_features=("feature","size"),missing=("train_missing","mean")).round(3))
if not cache_df.found.any():
    warnings.warn("No M8 acoustic/speech/phonetic caches found. Do NOT conclude these feature families add no signal.")

## 9 — Feature coverage, missingness, and train-test availability mismatch

A feature cannot help a model if it is absent for most children or systematically absent in the hidden test set. Inspect coverage before attributing weak gains to a representation ceiling.

In [ ]:
feature_quality=pd.DataFrame({"feature":columns,
    "family":[family[c] for c in columns],
    "train_missing":[X_train[c].isna().mean() for c in columns],
    "test_missing":[X_test[c].isna().mean() for c in columns]})
feature_quality["delta_missing"]=feature_quality.test_missing-feature_quality.train_missing
display(feature_quality.groupby("family").agg(count=("feature","size"),
     train_missing=("train_missing","mean"),test_missing=("test_missing","mean")).round(3))
fig,axs=plt.subplots(1,2,figsize=(14,5))
feature_quality.groupby("family")[["train_missing","test_missing"]].mean().plot.bar(
    ax=axs[0],rot=25,title="Missingness by family")
axs[0].set_ylabel("Missing fraction")
top=feature_quality.sort_values("train_missing",ascending=False).head(20).iloc[::-1]
axs[1].barh(top.feature.str.slice(0,50),top.train_missing)
axs[1].set(xlabel="Train missing fraction",title="Most incomplete features")
save_figure("03_feature_missingness","Feature and family missingness")
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(feature_quality.train_missing,feature_quality.test_missing,alpha=.7,s=18)
ax.plot([0,1],[0,1],"k--")
ax.set(xlabel="Train missing fraction",ylabel="Test missing fraction",title="Missingness drift train to hidden test")
save_figure("04_missingness_drift","Train/test feature-coverage mismatch")
display(feature_quality.sort_values("delta_missing",key=lambda z:z.abs(),ascending=False).head(20))

## 10 — Outcome-aware feature associations, descriptive only

Spearman coefficients help identify hypotheses and missing-signal families. **All train labels are visible here**, so these rankings must not be reused to select features inside the CV experiment below without nesting the selection.

In [ ]:
correlation_rows=[]
for col in columns:
    xx=pd.to_numeric(X_train[col],errors="coerce").to_numpy()
    valid=np.isfinite(xx)&np.isfinite(y)
    if valid.sum()<max(20,int(.35*len(y))) or len(np.unique(xx[valid]))<4:continue
    rho,p=spearmanr(xx[valid],y[valid])
    if np.isfinite(rho):
        correlation_rows.append({"feature":col,"family":family[col],"n":int(valid.sum()),
                                 "rho":float(rho),"p_exploratory":float(p),"abs_rho":abs(float(rho))})
correlations=pd.DataFrame(correlation_rows)
if not correlations.empty:
    display(correlations.sort_values("abs_rho",ascending=False).head(25).round(4))
    top=correlations.sort_values("abs_rho",ascending=False).head(25).iloc[::-1]
    fig,ax=plt.subplots(figsize=(11,8))
    ax.barh(top.feature.str.slice(0,56),top.rho)
    ax.set(xlabel="Spearman ρ (descriptive)",title="Strongest univariate outcome relationships")
    save_figure("05_feature_score_correlations","Descriptive feature-to-score relationships")
    display(correlations.groupby("family").abs_rho.agg(["count","median","max"]).sort_values("median",ascending=False).round(3))
else:print("No numeric feature associations available.")

## 11 — Fingerprint the experiment before running costly CV

The fingerprint includes the selected feature values, target, groups, configuration, and Git commit. Reusing old results is allowed only when this signature matches. Every executed run stores its signature on Drive.

In [ ]:
import dataclasses
CFG=AuditConfig(data_root=str(DATA_ROOT),output_root=str(OUTPUT_ROOT),profile_csv=PROFILE_CSV,
                quick=not FULL,folds=N_FOLDS,repeats=N_REPEATS,max_features=MAX_FEATURES,
                bootstrap=BOOTSTRAPS,seed=SEED,
                run_residual_probe=True,run_learning_curve=True,run_negative_control=True)
(OUT/"tables").mkdir(parents=True,exist_ok=True)
(OUT/"reports").mkdir(parents=True,exist_ok=True)
fingerprint=hashlib.sha256()
fingerprint.update(pd.util.hash_pandas_object(X_train.astype(str),index=False).to_numpy().tobytes())
fingerprint.update(np.asarray(y,dtype=np.float64).tobytes())
fingerprint.update(pd.util.hash_pandas_object(pd.Series(schools,dtype="string"),index=False).to_numpy().tobytes())
fingerprint.update(json.dumps(dataclasses.asdict(CFG),sort_keys=True,default=str).encode())
fingerprint.update(HEAD_SHA.encode())
RUN_SIGNATURE=fingerprint.hexdigest()
SIGNATURE_PATH=OUT/"reports"/"notebook_run_signature.json"
REUSABLE=False
if SIGNATURE_PATH.is_file() and (OUT/"tables"/"fold_metrics.csv").is_file() and (OUT/"M83_BOTTLENECK_REPORT.md").is_file():
    try:REUSABLE=json.loads(SIGNATURE_PATH.read_text()).get("signature")==RUN_SIGNATURE
    except (OSError,ValueError):pass
print("Experiment fingerprint:",RUN_SIGNATURE[:20])
print("Existing identical run:",REUSABLE, "| FORCE_RERUN:",FORCE_RERUN)
print("Private output path:",OUT)

## 12 — Run the leakage-safe M8.3 core tournament

Each feature family and feature-family dropout is compared on identical folds, against the mean predictor and screening-only baseline. The engine also writes train/validation gaps, student/school paired bootstraps, pair ordering, subgroup MAE, feature-only clusters, nested residual correction, learning curves and negative controls.

**Note:** full evaluation may require substantial CPU time; set `MODE="SMOKE"` for installation/IO validation first. Every result stays in Drive.

In [ ]:
if REUSABLE and not FORCE_RERUN:
    manifest=json.loads((OUT/"reports"/"diagnostic_manifest.json").read_text())
    print("Loaded identical existing diagnostic results.")
else:
    began=time.perf_counter()
    print("Beginning",MODE,"diagnostics on",len(y),"students and",len(columns),"features",flush=True)
    manifest=run_frame(X_train,y,schools,OUT,CFG)
    SIGNATURE_PATH.write_text(json.dumps({
        "signature":RUN_SIGNATURE,"commit":HEAD_SHA,"mode":MODE,
        "timestamp_utc":datetime.now(timezone.utc).isoformat(),
        "data_root":str(DATA_ROOT),"output_root":str(OUTPUT_ROOT),
        "profile_csv":PROFILE_CSV,"config":dataclasses.asdict(CFG)},indent=2))
    print("Completed in",round(time.perf_counter()-began,1),"seconds.")
print("Main report:",OUT/"M83_BOTTLENECK_REPORT.md")
for note in manifest.get("notes",[]):print(" •",note)

## 13 — Load every M8.3 result, verify OOF coverage, and validate school isolation

Before interpreting figures, check that each model has one prediction per student per repetition, numeric predictions are finite, and no held-out school appears in its corresponding training fold.

In [ ]:
names=["fold_metrics","oof_predictions_PRIVATE","paired_bootstrap","pair_gap_diagnostics",
       "pair_overall","subgroup_errors","residual_probe","negative_control",
       "learning_curve","feature_clusters","generalization_gaps"]
TABLES={}
for name in names:
    path=OUT/"tables"/(name+".csv")
    TABLES[name]=pd.read_csv(path) if path.is_file() else pd.DataFrame()
    print(f"{name:32s} {len(TABLES[name]):7d} rows", "✓" if path.exists() else "(unavailable)")
folds=TABLES["fold_metrics"]
oof=TABLES["oof_predictions_PRIVATE"]
assert not folds.empty and not oof.empty, "Critical M8.3 fold/OOF files absent"
assert oof.pred.notna().all() and np.isfinite(oof.pred).all()
assert set(oof.row.astype(int))==set(range(len(y))), "Missing some students from OOF"
for (split,repeat,model),g in oof.groupby(["split","repeat","model"]):
    assert len(g)==len(y) and g.row.is_unique, f"OOF mismatch {split}/{repeat}/{model}"
if "grouped" in set(oof.split):
    school_array=np.asarray(schools,dtype=str)
    for (rep,fold),g in oof[(oof.split=="grouped")&(oof.model=="all")].groupby(["repeat","fold"]):
        val_idx=g.row.to_numpy(int)
        train_idx=np.setdiff1d(np.arange(len(y)),val_idx)
        assert not set(school_array[val_idx])&set(school_array[train_idx]),"School-disjoint validation violated"
print("PASS: OOF coverage, predictions, and group disjointness.")
display(Markdown((OUT/"M83_BOTTLENECK_REPORT.md").read_text(encoding="utf8")))

## 14 — Feature-family ablation tournament

The critical question: **does adding speech, acoustic, phonetic and behavioral information reduce MAE beyond screening?** Read the random and school-disjoint scores separately. Uncertainty should be evaluated using paired bootstrap, not just mean fold ranking.

In [ ]:
scoreboard=(folds.groupby(["split","model"],as_index=False)
           .agg(mae=("mae","mean"),rmse=("rmse","mean"),r2=("r2","mean"),
                train_mae=("train_mae","mean"),n_folds=("fold","size")))
display(scoreboard.sort_values(["split","mae"]).round(4))
base_model="screening_only" if "screening_only" in set(scoreboard.model) else "mean"
fig,axs=plt.subplots(1,2,figsize=(14,6))
for ax,split in zip(axs,("random","grouped")):
    z=scoreboard[scoreboard.split==split].sort_values("mae").head(16).iloc[::-1]
    if z.empty:ax.text(.5,.5,"Unavailable",ha="center")
    else:ax.barh(z.model,z.mae)
    ax.set(title=split+" CV: ablation scoreboard",xlabel="Mean holdout MAE")
save_figure("06_feature_ablation","Matched-fold feature ablation scoreboard")
all_mae=scoreboard[scoreboard.model=="all"].set_index("split").mae
drop=scoreboard[scoreboard.model.str.startswith("without_")].copy()
if not drop.empty:
    drop["damage_when_removed"]=[v-all_mae.get(s,np.nan) for s,v in zip(drop.split,drop.mae)]
    display(drop[["split","model","mae","damage_when_removed"]]
            .sort_values("damage_when_removed",ascending=False).round(5))
for split in ("random","grouped"):
    z=scoreboard[scoreboard.split==split].set_index("model")
    if {"all",base_model}.issubset(z.index):
        print(f"{split}: {base_model} minus all features = {z.at[base_model,'mae']-z.at['all','mae']:+.5f} MAE")

## 15 — Student and school paired-bootstrap uncertainty

Positive gain means the full feature model improves MAE. Resampling school groups preserves within-school dependence; the 95% percentile interval is **conditional on these validation splits**, not a substitute for a completely independent holdout.

In [ ]:
bootstrap=TABLES["paired_bootstrap"]
if not bootstrap.empty:
    ci=(bootstrap.groupby(["unit","repeat"]).mae_improvement
       .agg(median="median",lo=lambda x:x.quantile(.025),hi=lambda x:x.quantile(.975))).reset_index()
    display(ci.round(5))
    fig,ax=plt.subplots(figsize=(9,4))
    for row in ci.itertuples():
        line=0 if row.unit=="student" else 1
        yloc=line+.1*row.repeat
        ax.errorbar(row.median,yloc,xerr=[[row.median-row.lo],[row.hi-row.median]],
                    fmt="o",capsize=4)
    ax.axvline(0,color="black",ls="--")
    ax.set(yticks=[0,1],yticklabels=["Student bootstrap","School bootstrap"],
           xlabel=f"MAE gain: {base_model} minus all",title="Paired OOF uncertainty")
    save_figure("07_bootstrap_uncertainty","Uncertainty in incremental feature value")
else:print("Paired bootstrap not available.")

## 16 — Overfitting versus school-transfer failure

Large training-validation error gaps suggest variance/overfit; a large gap between random-student and group-disjoint evaluations suggests domain transfer difficulties. The two validation schemes are distinct estimands, not experimentally randomized school effects.

In [ ]:
focus=folds[folds.model=="all"].copy()
focus["gap_mae"]=focus.mae-focus.train_mae
display(focus.groupby("split")[["train_mae","mae","gap_mae","rmse","r2"]]
        .agg(["mean","std"]).round(4))
fig,axs=plt.subplots(1,2,figsize=(12,4))
focus.groupby("split")[["train_mae","mae"]].mean().plot.bar(ax=axs[0],rot=0)
axs[0].set(title="Training vs validation MAE",ylabel="MAE")
rng=np.random.default_rng(SEED)
for i,(split,g) in enumerate(focus.groupby("split")):
    axs[1].scatter(np.full(len(g),i)+rng.normal(0,.035,len(g)),g.gap_mae,alpha=.7,label=split)
axs[1].axhline(0,color="black",ls="--")
axs[1].set(xticks=list(range(focus.split.nunique())),
           xticklabels=sorted(focus.split.unique()),ylabel="Validation MAE − Train MAE",
           title="Generalization gap by fold")
axs[1].legend()
save_figure("08_train_validation_gaps","Generalization and unseen-school gaps")

## 17 — Out-of-fold calibration, regression to the mean, and heteroscedasticity

Never pool repeated-CV predictions as independent students. Plot the **first repeat** of the full model, then inspect where true-score range compression and systematic error occur.

In [ ]:
def oof_view(split="random",model="all",repeat=0):
    return oof[(oof.split==split)&(oof.model==model)&(oof.repeat==repeat)].sort_values("row").copy()
d=oof_view()
assert len(d)==len(y)
d["error"]=d.target-d.pred
d["abs_error"]=d.error.abs()
fig,ax=plt.subplots(2,2,figsize=(12,10))
low=min(d.target.min(),d.pred.min());high=max(d.target.max(),d.pred.max())
ax[0,0].scatter(d.target,d.pred,s=17,alpha=.6)
ax[0,0].plot([low,high],[low,high],"k--")
ax[0,0].set(xlabel="True score",ylabel="OOF prediction",title="Observed vs predicted")
ax[0,1].scatter(d.pred,d.error,s=17,alpha=.6);ax[0,1].axhline(0,color="black",ls="--")
ax[0,1].set(xlabel="Predicted score",ylabel="True − predicted",title="Residuals vs prediction")
ax[1,0].hist(d.error,bins=24,edgecolor="white")
ax[1,0].axvline(0,color="black",ls="--")
ax[1,0].set(xlabel="OOF error",ylabel="Students",title="Error distribution")
ax[1,1].scatter(d.target,d.abs_error,s=17,alpha=.6)
ax[1,1].set(xlabel="True score",ylabel="Absolute error",title="Error by true score")
save_figure("09_calibration_and_residuals","Out-of-fold calibration and error geometry")
print("OOF MAE:",round(float(d.abs_error.mean()),5),
      " | Mean bias:",round(float(d.error.mean()),5),
      " | Pred/label std:",round(float(d.pred.std()/d.target.std()),4))

## 18 — Score ranges, schools, and feature-condition failure maps

Score quartiles are known only for labeled data; they must not become a deployment-time gating signal. School labels are masked in plots. Train-feature-stratified MAE is descriptive; future targeting needs independent validation.

In [ ]:
subgroups=TABLES["subgroup_errors"]
if not subgroups.empty:
    dims=[x for x in ("score","prediction","school") if x in set(subgroups.dimension)]
    fig,axs=plt.subplots(1,len(dims),figsize=(5*len(dims),4))
    if len(dims)==1:axs=[axs]
    for ax,dim in zip(axs,dims):
        z=subgroups[subgroups.dimension==dim].sort_values("mae",ascending=False).head(20)
        labels=["G"+str(i+1) for i in range(len(z))] if dim=="school" else [str(i+1) for i in range(len(z))]
        ax.bar(range(len(z)),z.mae)
        ax.set(xticks=list(range(len(z))),xticklabels=labels,title=dim+" error patterns",ylabel="MAE")
        ax.tick_params(axis="x",rotation=65)
    save_figure("10_error_subgroups","Failure concentration by score, prediction, and school")
    other=subgroups[~subgroups.dimension.isin(("score","prediction","school"))]
    if not other.empty:
        r=other.groupby("dimension").mae.agg(["min","max","count"])
        r=r[r["count"]>=2].copy();r["range"]=r["max"]-r["min"]
        display(r.sort_values("range",ascending=False).head(20).round(4))
else:print("No subgroup diagnostic returned.")

## 19 — Pairwise ranking difficulty by actual score gap

The same child can occur in hundreds of pairs, so pair metrics are **descriptive, not IID binomial accuracy estimates**. The gap-conditioned curve shows whether low MAE masks serious ordering failures.

In [ ]:
pairs=TABLES["pair_gap_diagnostics"]
pair_overall=TABLES["pair_overall"]
if not pairs.empty:
    display(pairs.round(4))
    display(pair_overall)
    fig,ax=plt.subplots(figsize=(9,4))
    ax.plot(pairs.median_true_gap,pairs.rank_accuracy,"o-",lw=2)
    ax.axhline(.5,ls="--",color="gray")
    ax.set(xlabel="Median true score difference",ylabel="Correct pair ordering",
           ylim=(0,1),title="Ranking difficulty: OOF comparisons")
    save_figure("11_pair_ordering","Pairwise ordering versus label distance")
else:print("No pairwise diagnostic available.")

## 20 — What does the model consistently miss? Descriptive residual associations

Test which original feature values are associated with OOF errors. These *hypotheses* may guide new representations, but selecting a new model using these observations and then testing on the same folds would be optimistic.

In [ ]:
residual_rows=[]
for col in columns:
    v=pd.to_numeric(X_train.iloc[d.row.astype(int).to_numpy()][col],errors="coerce").to_numpy()
    good=np.isfinite(v)&np.isfinite(d.error.to_numpy())
    if good.sum()<max(20,int(.4*len(d))) or len(np.unique(v[good]))<5:continue
    rho,_=spearmanr(v[good],d.error.to_numpy()[good])
    if np.isfinite(rho):
        residual_rows.append({"feature":col,"family":family[col],"n":int(good.sum()),
                              "residual_rho":float(rho),"abs_rho":abs(float(rho))})
residual_associations=pd.DataFrame(residual_rows)
if not residual_associations.empty:
    display(residual_associations.sort_values("abs_rho",ascending=False).head(25).round(4))
    z=residual_associations.sort_values("abs_rho",ascending=False).head(24).iloc[::-1]
    fig,ax=plt.subplots(figsize=(11,8))
    ax.barh(z.feature.str.slice(0,58),z.residual_rho)
    ax.set(xlabel="Spearman correlation with residual (descriptive)",
           title="Feature correlates of OOF prediction error")
    save_figure("12_residual_correlations","Residual-feature descriptive associations")

## 21 — Near-neighbor label contradictions

Create a compact **target-blind screening representation**, find nearby student pairs, and quantify score discordance. Large near-neighbor differences can indicate missing linguistic signal, domain effects or weak distance geometry. They are **not** a validated annotation-noise estimate.

In [ ]:
neighbor_pairs=pd.DataFrame()
if RUN_NEIGHBORS and len(y)>=15:
    from sklearn.decomposition import PCA
    from sklearn.neighbors import NearestNeighbors
    screening=[c for c in columns if family[c]=="screening"]
    if len(screening)>=2:
        Z=FoldEncoder().fit(X_train[screening]).transform(X_train[screening])
        if Z.shape[1]>=2:
            latent=PCA(n_components=min(8,Z.shape[1],len(y)-1),random_state=SEED).fit_transform(Z)
            D,I=NearestNeighbors(n_neighbors=min(6,len(y))).fit(latent).kneighbors(latent)
            pool={}
            for i in range(len(y)):
                for j in range(1,I.shape[1]):
                    k=int(I[i,j])
                    if k==i:continue
                    a,b=sorted((i,k))
                    pool[(a,b)]=min(pool.get((a,b),float("inf")),float(D[i,j]))
            neighbor_pairs=pd.DataFrame([{"row_a":a,"row_b":b,"distance":dist,
                                           "score_gap":abs(float(y[a]-y[b]))}
                                           for (a,b),dist in pool.items()])
            if not neighbor_pairs.empty:
                display(neighbor_pairs[["distance","score_gap"]].describe(percentiles=[.1,.5,.9,.95]).round(4))
                fig,ax=plt.subplots(figsize=(9,4))
                ax.scatter(neighbor_pairs.distance,neighbor_pairs.score_gap,s=18,alpha=.4)
                ax.set(xlabel="Screening embedding distance",ylabel="Absolute reading-score gap",
                       title="Score disagreement among nearby screening profiles")
                save_figure("13_neighbor_contradictions","Nearest-neighbor score discrepancies")
    else:print("No screening features for neighbor representation.")
else:print("Neighbor analysis disabled or insufficient sample count.")

## 22 — Nested residual correction: is there learnable signal left?

The residual model is trained only on **inner-fold cross-fitted training residuals**, then evaluated on untouched outer folds. If it cannot reliably improve even on school-held-out folds, adding a mixture-of-experts is unlikely to be a useful immediate investment.

In [ ]:
probe=TABLES["residual_probe"]
if not probe.empty:
    display(probe.groupby("split")[["base_mae","residual_corrected_mae","mae_improvement"]]
            .agg(["mean","std"]).round(5))
    fig,axs=plt.subplots(1,2,figsize=(12,4))
    probe.groupby("split")[["base_mae","residual_corrected_mae"]].mean().plot.bar(
        ax=axs[0],rot=0,title="Nested residual correction");axs[0].set_ylabel("MAE")
    for split,g in probe.groupby("split"):
        axs[1].scatter(range(len(g)),g.mae_improvement,label=split)
    axs[1].axhline(0,color="black",ls="--");axs[1].legend()
    axs[1].set(xlabel="Validation fold",ylabel="Positive = better",
               title="Residual correction gains")
    save_figure("14_residual_probe","Fold-safe residual predictability")
else:print("Residual probe unavailable.")

## 23 — Learning curves and null-label control

A decreasing validation error with more training students supports a **data scarcity hypothesis**; a true-label model that fails to beat shuffled training labels raises a pipeline/data-join warning. Neither experiment establishes a precise Bayes error floor.

In [ ]:
curve=TABLES["learning_curve"]; negative=TABLES["negative_control"]
fig,axs=plt.subplots(1,2,figsize=(12,4))
if not curve.empty:
    agg=curve.groupby("train_fraction")[["train_mae","valid_mae"]].agg(["mean","std"])
    for model in ("train_mae","valid_mae"):
        axs[0].errorbar(agg.index,agg[(model,"mean")],yerr=agg[(model,"std")].fillna(0),
                        marker="o",capsize=3,label=model)
    axs[0].legend();axs[0].set(xlabel="Fraction of available training set",ylabel="MAE",title="Learning curve")
    display(curve.groupby("train_fraction")[["train_n","train_mae","valid_mae"]].mean().round(4))
else:axs[0].text(.5,.5,"Unavailable",ha="center")
if not negative.empty:
    vals=negative[["true_mae","null_mae"]].mean()
    axs[1].bar(["Real labels","Shuffled labels"],vals.to_numpy())
    axs[1].set(ylabel="MAE",title="Permutation negative control")
    print("Real/shuffled MAE:",vals.round(4).to_dict())
else:axs[1].text(.5,.5,"Unavailable",ha="center")
save_figure("15_learning_curve_and_null","Data scale and null-label controls")

## 24 — Fold-local feature clusters and held-out error patterns

Unsupervised clusters are fitted from **outer training features** and associated with errors only on validation children. Cluster numbers are *local to each fold* and must never be interpreted as stable phenotypes without a separate alignment and stability experiment.

In [ ]:
cluster=TABLES["feature_clusters"]
if not cluster.empty:
    display(cluster.sort_values("mae",ascending=False).head(25).round(4))
    top=cluster.sort_values("mae",ascending=False).head(25).iloc[::-1]
    titles=[f"fold {int(r.fold)} · c{int(r.cluster)} · n={int(r.n)}" for r in top.itertuples()]
    fig,ax=plt.subplots(figsize=(10,8))
    ax.barh(titles,top.mae);ax.set(xlabel="Held-out cluster MAE",title="Fold-local feature-regime failure")
    save_figure("16_fold_local_cluster_errors","Outcome-blind fold-local clusters with OOF error")
else:print("No feature-cluster metrics available.")

## 25 — Train/test covariate shift: numeric standardized differences

This is an **unsupervised diagnostic**. The competition test labels are not accessed. Strong shifts imply that random CV might be optimistic for hidden-test performance, but distribution shift alone does not prove target or concept shift.

In [ ]:
drift_rows=[]
for col in columns:
    a=pd.to_numeric(X_train[col],errors="coerce").replace([np.inf,-np.inf],np.nan).dropna()
    b=pd.to_numeric(X_test[col],errors="coerce").replace([np.inf,-np.inf],np.nan).dropna()
    if len(a)<10 or len(b)<10:continue
    spread=float(np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2))
    if not np.isfinite(spread) or spread<1e-9:continue
    shift=float((b.mean()-a.mean())/spread)
    drift_rows.append({"feature":col,"family":family[col],"std_mean_delta":shift,
                       "abs_shift":abs(shift),"train_n":len(a),"test_n":len(b)})
drift=pd.DataFrame(drift_rows)
if not drift.empty:
    display(drift.sort_values("abs_shift",ascending=False).head(25).round(4))
    z=drift.sort_values("abs_shift",ascending=False).head(22).iloc[::-1]
    fig,ax=plt.subplots(figsize=(11,7))
    ax.barh(z.feature.str.slice(0,55),z.std_mean_delta)
    ax.set(xlabel="Test − train: standardized mean difference",title="Numeric feature shift")
    save_figure("17_train_test_covariate_shift","Train versus test numeric feature drift")
else:print("Not enough continuous feature coverage to assess standardized differences.")

## 26 — Can a classifier distinguish train from hidden test?

A fold-fitted domain discriminator uses **features only** and estimates how easy it is to distinguish the two distributions. AUC far above 0.5 suggests a shift; this is not a substitute for competition evaluation. All preprocessing stays within CV folds.

In [ ]:
domain_auc=None
domain_auc_folds=[]
if RUN_DRIFT_AUC and min(len(X_train),len(X_test))>=30:
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    numeric=[c for c in columns if pd.api.types.is_numeric_dtype(X_train[c])]
    if numeric:
        domain_X=pd.concat([X_train[numeric],X_test[numeric]],ignore_index=True).replace([np.inf,-np.inf],np.nan)
        domain_y=np.r_[np.zeros(len(X_train),dtype=int),np.ones(len(X_test),dtype=int)]
        domain_model=Pipeline([
            ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
            ("scale",StandardScaler()),
            ("classifier",LogisticRegression(C=.1,max_iter=1500))])
        fold_auc=cross_val_score(domain_model,domain_X,domain_y,
             cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED),
             scoring="roc_auc",n_jobs=1)
        domain_auc_folds=fold_auc.tolist()
        domain_auc=float(fold_auc.mean())
        print("Train/test domain AUC",round(domain_auc,4),"per fold",np.round(fold_auc,4))
        fig,ax=plt.subplots(figsize=(6,4))
        ax.bar(range(1,len(fold_auc)+1),fold_auc)
        ax.axhline(.5,ls="--",color="black")
        ax.set(xlabel="CV fold",ylabel="Train vs hidden-test AUC",
               ylim=(0,1),title="How distinguishable are the feature distributions?")
        save_figure("18_domain_classifier","CV train/test discriminator")
    else:print("No numeric features for domain classifier.")
else:print("Domain classification disabled or too few samples.")

## 27 — Evidence scorecard and decision hypotheses

Prioritize next experiments using **matched-fold effects and nested residual evidence**. A small or uncertain uplift should trigger investigation, not another complex ensemble. Distinguish real model evidence from exploratory plots.

In [ ]:
EVIDENCE=[]
s=scoreboard.set_index(["split","model"])
for split in ("random","grouped"):
    if (split,"all") in s.index and (split,base_model) in s.index:
        uplift=float(s.at[(split,base_model),"mae"]-s.at[(split,"all"),"mae"])
        EVIDENCE.append(("feature_increment_"+split,
                         f"All vs {base_model}: MAE improvement {uplift:+.4f}. "+
                         ("Positive but requires replication." if uplift>0 else "New features do not help on this split.")))
if not bootstrap.empty:
    b=bootstrap[(bootstrap.unit=="group")&(bootstrap.repeat==0)].mae_improvement
    if len(b):
        lo,hi=np.quantile(b,[.025,.975])
        EVIDENCE.append(("incremental_uncertainty",f"School-group bootstrap interval [{lo:+.4f}, {hi:+.4f}]; "+
                         ("includes zero." if lo<=0<=hi else "excludes zero conditional on this CV.")))
if ("random","all") in s.index and ("grouped","all") in s.index:
    a=float(s.at[("random","all"),"mae"]);b=float(s.at[("grouped","all"),"mae"])
    EVIDENCE.append(("school_generalization",f"Random CV MAE {a:.4f} vs held-out-school CV MAE {b:.4f}. Different estimands."))
if not probe.empty:
    for split,g in probe.groupby("split"):
        EVIDENCE.append(("nested_residual_"+split,
                         f"Mean MAE correction gain {g.mae_improvement.mean():+.4f}; positive folds {(g.mae_improvement>0).sum()}/{len(g)}."))
if not curve.empty:
    cs=curve.groupby("train_fraction").valid_mae.mean().sort_index()
    EVIDENCE.append(("learning_curve",f"Smallest-data MAE {cs.iloc[0]:.4f} → largest-data MAE {cs.iloc[-1]:.4f}; "+
                     ("more data may help." if cs.iloc[-1]<cs.iloc[0] else "no observed scaling gain.")))
if not negative.empty and negative.true_mae.mean()>=negative.null_mae.mean():
    EVIDENCE.append(("critical_negative_control","True-label model did not beat shuffled-training-label model: audit data/joins."))
if domain_auc is not None:
    EVIDENCE.append(("test_covariate_shift",f"Train/test discriminator AUC {domain_auc:.3f} (unsupervised diagnostic)."))
if not PROFILE_CSV:
    EVIDENCE.append(("missing_profile","M7 student-profile baseline is missing. Fix Drive paths before interpreting feature-family gains."))
for info in cache_status:
    if not info["found"]:
        EVIDENCE.append(("missing_"+info["family"]+"_features",
                         f"{info['family']} cache not found in {OUTPUT_ROOT}/tables; its incremental value is untested."))
if not EVIDENCE:
    EVIDENCE.append(("data_diagnosis","No conclusive bottleneck rule triggered; inspect raw tables and validate externally."))
scorecard=pd.DataFrame(EVIDENCE,columns=["hypothesis","evidence"])
display(scorecard)
print("These are evidence-based hypotheses, NOT proof of a numeric Bayes/annotation-noise ceiling.")

## 28 — Save the exhaustive research atlas, figures, tables and reproducibility manifest

All outputs stay under `OUTPUT_ROOT/m83_bottleneck` on your mounted **Google Drive**, separate from the GitHub repository. No child-level identifiers, raw school IDs or hidden test labels appear in the extended Markdown/HTML reports.

In [ ]:
import html
REPORTS=OUT/"reports"
REPORTS.mkdir(parents=True,exist_ok=True)
def export_csv(name,df):
    if not df.empty:
        path=OUT/"tables"/name
        df.to_csv(path,index=False)
        return str(path)
    return None
export_csv("notebook_feature_quality.csv",feature_quality)
export_csv("notebook_score_correlations_DESCRIPTIVE.csv",correlations)
export_csv("notebook_residual_correlations_DESCRIPTIVE.csv",residual_associations)
export_csv("notebook_train_test_drift.csv",drift)
export_csv("notebook_nearest_neighbor_pairs_PRIVATE.csv",neighbor_pairs)
export_csv("notebook_evidence_scorecard.csv",scorecard)
export_csv("notebook_figures.csv",pd.DataFrame(FIGURES))
report_lines=[
    "# WildSpeech M8.3 — Exhaustive Bottleneck Research",
    "",f"- Mode: **{MODE}**",f"- GitHub commit: `{HEAD_SHA}`",
    f"- Labeled students: **{len(y)}**",f"- Hidden-test students: **{len(X_test)}**",
    f"- Selected features: **{len(columns)}**",f"- Schools: **{len(set(schools))}**",
    "- M7 profile: "+("present" if PROFILE_CSV else "**MISSING**"),
    "- Feature caches: "+", ".join(x["family"]+"="+("found" if x["found"] else "MISSING") for x in cache_status),
    "","## Diagnostics and next decision",""]
report_lines += [f"- **{name}:** {finding}" for name,finding in EVIDENCE]
report_lines += ["","## Scientific boundaries","",
    "- Feature/outcome associations, neighbor discrepancy and clusters are descriptive only.",
    "- Out-of-fold metrics are validation evidence, not official competition results.",
    "- School-disjoint CV answers a different transfer question from random CV.",
    "- Pairwise accuracy uses dependent pairs; no IID pair confidence intervals.",
    "- Residual correction uses training-only inner OOF residuals and untouched outer validation.",
    "- Annotation-noise ceilings require repeated labeling, not nearest-neighbor disagreement.",
    "- All raw student data and OOF predictions remain private.",
    "","## Figure index",""]
report_lines += [f"- {x['title']}: `{x['path']}`" for x in FIGURES]
(REPORTS/"M83_DEEP_DIVE.md").write_text("\n".join(report_lines),encoding="utf8")
html_parts=["<!doctype html><html lang='en'><head><meta charset='utf-8'>",
    "<title>WildSpeech M8.3 Research Atlas</title><style>",
    "body{font:16px system-ui,sans-serif;color:#203042;max-width:1150px;margin:40px auto;padding:0 24px;line-height:1.6}",
    "header{background:#eef2f6;padding:20px;border-radius:12px}",
    "figure{margin:30px 0;border:1px solid #dce4ed;border-radius:12px;padding:18px}",
    "img{max-width:100%;height:auto}small{color:#586575}</style></head><body>",
    f"<header><h1>WildSpeech M8.3 · Exhaustive Bottleneck Atlas</h1><p>{html.escape(MODE)} · "+
    f"{len(y)} labeled students · {len(columns)} features · code {html.escape(HEAD_SHA[:12])}</p>",
    "<p>Research validation only. Do not infer an official leaderboard gain from synthetic or descriptive results.</p></header>",
    "<h2>Bottleneck evidence</h2><ul>"]
html_parts += ["<li><b>"+html.escape(k)+"</b>: "+html.escape(v)+"</li>" for k,v in EVIDENCE]
html_parts += ["</ul><h2>Figures</h2>"]
for info in FIGURES:
    html_parts.append("<figure><h3>"+html.escape(info["title"])+
                      "</h3><img loading='lazy' src='"+html.escape(info["path"])+
                      "' alt='"+html.escape(info["title"])+"'></figure>")
html_parts.append("<small>Private student-level outputs stay in Drive. This atlas contains aggregate evidence only.</small></body></html>")
(OUT/"M83_ATLAS.html").write_text("\n".join(html_parts),encoding="utf8")
notebook_manifest={
    "mode":MODE,"commit":HEAD_SHA,"signature":RUN_SIGNATURE,
    "data_root":str(DATA_ROOT),"output_root":str(OUT),
    "profile_source":PROFILE_CSV or None,"caches":cache_status,
    "n_students":len(y),"n_test":len(X_test),"n_schools":len(set(schools)),
    "n_features":len(columns),"n_figures":len(FIGURES),
    "domain_auc":domain_auc,"domain_auc_folds":domain_auc_folds,
    "timestamp_utc":datetime.now(timezone.utc).isoformat()}
(REPORTS/"M83_NOTEBOOK_MANIFEST.json").write_text(json.dumps(notebook_manifest,indent=2,default=str),encoding="utf8")
print("Saved",len(FIGURES),"research figures")
print("Main atlas:",OUT/"M83_ATLAS.html")
print("Extended report:",REPORTS/"M83_DEEP_DIVE.md")
print("Figure index:",OUT/"tables"/"notebook_figures.csv")

## 29 — Final gate: what experiment should come next?

Read `M83_DEEP_DIVE.md`, the saved `M83_ATLAS.html`, the model ablation table and the matched-bootstrap intervals. Choose **only one** intervention for a new independent validation round:

- **Missing baseline/cache:** fix profile and acoustic data coverage *before* model changes.
- **Large train-validation gap:** simplify or regularize the model, re-evaluate by school.
- **Good random CV but poor school CV:** study school/recording confounders and domain invariance.
- **Complete audio cache but no incremental gain:** test qualitatively different frozen speech encoders or transcript features, not more summary statistics.
- **Nested residual correction helps on untouched school folds:** prototype conservative mixture-of-experts with an independently selected gate.
- **No feature or residual signal:** revisit score measurement and task definition; repeated annotations are necessary to estimate label uncertainty.

**For exhaustive official-data results:** set `MODE="FULL"` at the top and execute the complete notebook. The notebook runs in Colab with Drive mounted and saves everything under `MyDrive/WildSpeech_M8/m83_bottleneck` (or your discovered cache root).